In [4]:
%%time

from google.colab import drive

import os

# if not mounted
if not os.path.exists('/content/drive'):
  drive.mount('/content/drive')

CPU times: user 647 µs, sys: 36 µs, total: 683 µs
Wall time: 623 µs


In [5]:
%%time

from sqlalchemy import create_engine

db_name = 'nfs_m.db'

db_loc = f'sqlite:///drive/MyDrive/colab/banshee/data/{db_name}'
engine = create_engine(db_loc)

import pandas as pd

vid_stats_df = pd.read_sql_table('video_statistics', engine)

vid_stats_rename_map = {
    'video_published_at': 'timestamp',
    'video_description': 'description',
    'video_title': 'title',
    'video_tags': 'tags',
    'channel_id': 'channel',
    'video_duration': 'duration',
    'video_topic_categories': 'video_topics',
    'channel_topic_categories': 'channel_topics',
    'video_caption': 'caption',
    'video_view_count': 'views',
}

vid_stats_df.rename(columns=vid_stats_rename_map, inplace=True)

vid_stats_drop_cols = ['video_like_count', 'video_comment_count',
                       'video_licensed_content']

vid_stats_df.drop(columns=vid_stats_drop_cols, inplace=True)

vid_stats_df.set_index('video_id', inplace=True)

vid_stats_df['views'] = vid_stats_df['views'].astype(int)

vid_stats_df.sort_values('timestamp', inplace=True)

vid_stats_df['timestamp'] = pd.to_datetime(vid_stats_df['timestamp'])
vid_stats_df['timestamp'] = vid_stats_df['timestamp'].dt.tz_localize(None)
vid_stats_df['timestamp'] -= pd.Timestamp("1970-01-01")
vid_stats_df['timestamp'] //= pd.Timedelta('1s')

vid_stats_df = vid_stats_df[~vid_stats_df.index.duplicated(keep='first')]

vid_stats_df.shape

CPU times: user 16.8 s, sys: 3.24 s, total: 20 s
Wall time: 24.8 s


(1150311, 10)

In [18]:
import spacy

# Load spaCy model
nlp = spacy.load("en_core_web_sm")

def noun_tokenizer(text):
    doc = nlp(text)
    tokens = []
    for chunk in doc.noun_chunks:
        tokens.append(chunk.text)
    return tokens

def all_tokenizer(text):
    doc = nlp(text)
    tokens = []

    # Add specific phrase detection for "Need for Speed" or other patterns if required
    phrase = "Need for Speed"
    if phrase in text:
        tokens.append(phrase)
        # Remove individual words of the phrase from further tokenization
        text = text.replace(phrase, "")

    # Process remaining text and extract all tokens (not just noun chunks)
    doc = nlp(text)
    for token in doc:
        if not token.is_punct:  # Skip punctuation tokens
            tokens.append(token.text)

    return tokens

def named_entity_tokenizer(text):
    doc = nlp(text)
    named_entities = []

    # Extract and append named entities to the list
    for entity in doc.ents:
        named_entities.append(entity.text.strip())

    return named_entities

import re

def normalize_text(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    text = text.strip()
    return text

OSError: [E050] Can't find model 'en_core_web_lg'. It doesn't seem to be a Python package or a valid path to a data directory.

In [17]:
df = vid_stats_df.copy()

from tqdm.notebook import tqdm

tqdm.pandas()

drop_cols = ['channel_title', 'caption']

df.drop(columns=drop_cols, inplace=True)

df = df[df['duration'] >= 480]

df = df[df['views'] >= 1000000]

df['title_ner'] = df['title'].progress_apply(named_entity_tokenizer)

df

  0%|          | 0/10369 [00:00<?, ?it/s]

,timestamp,channel,title,description,tags,duration,views,video_topics,title_ner
video_id,,,,,,,,,
vFeGAxRLELI,1261703062,UC1SjQSYp44B-D3FSkDCVeoQ,NFSC Career Secret Tutorial,"Subscribe and enjoy! First, I would like to wi...","nfsc, nfs, corvette, zo6, need, for, speed, ca...",595,1693469,"Action game, Racing video game, Video game cul...",[]
Yxwgg35Bt5E,1290326108,UC7FvHJEkTuwoi2pFsVideFA,Need For Speed Hot Pursuit 2010 PC Gameplay P...,Need For Speed videos https://youtube.com/play...,"Need, For, Speed, Hot, Pursuit, 2010, PC, Game...",793,1558675,"Action game, Racing video game, Video game cul...","[2010, 4]"
SHAWqkFwOCs,1297005115,UCuneWYkXOatJ4JH5QBXccBQ,Test Drive Unlimited 2 - All 10 Wreck Cars Loc...,Like this video? Don't forget to click thumbs ...,"test, drive, unlimited, tdu, tdu2, treasure, h...",482,1139728,"Action-adventure game, Racing video game, Simu...","[2 - All, 10]"
xM2PMWSH-F4,1326912110,UC-RyjxKstSxFtC7Xc4HHT_A,Need for Speed Underground 2 - Final Race,"Reiji ""The Showoff Samurai"" Maeda putting Cale...","need, for, speed, underground, nfsu2, mitsubis...",857,2451622,"Action game, Racing video game, Video game cul...",[]
3I6BLurNt5g,1331466471,UC-RyjxKstSxFtC7Xc4HHT_A,Need for Speed: Most Wanted - Blacklist #1: Razor,"vs Clarence ""RAZOR"" Callahan","need, for, speed, most, wanted, nfsmw, mitsubi...",517,1766500,"Action game, Racing video game, Video game cul...",[]
...,...,...,...,...,...,...,...,...,...
iSvNZ_7s4yY,1720285833,UCNvzD7Z-g64bPXxGzaQaa4g,25 Hardest Easter Eggs That TOOK YEARS TO DISC...,Video game secrets and easter eggs are often t...,"hardest easter eggs, top easter eggs, ps4 east...",1823,1410299,"Action-adventure game, Action game, Role-playi...",[25]
yH2LH_h2sFM,1720357256,UC0A86RKLCqTEUna3hPlEpzg,FROZEN Full Movie 2024: Elsa Frozen | Kingdom ...,FROZEN Full Movie 2024: Elsa Frozen | Kingdom ...,"frozen full movie, frozen, full, movie, 2024, ...",7218,3267059,"Action-adventure game, Action game, Video game...","[FROZEN Full Movie, 2024, Elsa Frozen, Kingdom..."
Ely2xdA0gsA,1721566822,UC0A86RKLCqTEUna3hPlEpzg,THE ELDER SCROLLS Full Movie 2024: Dragon | Su...,THE ELDER SCROLLS Full Movie 2024: Werewolf vs...,"the elder scrolls full movie, the elder scroll...",7214,1418257,"Action-adventure game, Action game, Role-playi...","[2024, English]"


# Nope

In [94]:
%%time

from sklearn.feature_extraction.text import TfidfVectorizer

from tqdm.notebook import tqdm

tqdm.pandas()

doc_df = vid_stats_df

documents = doc_df['title'].to_list()

print(len(documents))

# Use custom tokenizer with TF-IDF
tfidf_vectorizer = TfidfVectorizer(tokenizer=named_entity_tokenizer,
                                   max_df=0.5, min_df=8)

tfidf_matrix = tfidf_vectorizer.fit_transform(documents)

tfidf_df = pd.DataFrame(tfidf_matrix.toarray(),
                        columns=tfidf_vectorizer.get_feature_names_out(),
                        index=doc_df.index)

tfidf_df.shape

1150311


/usr/local/lib/python3.10/dist-packages/sklearn/feature_extraction/text.py:525: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


CPU times: user 1h 37min 24s, sys: 14.9 s, total: 1h 37min 38s
Wall time: 1h 37min 24s


(1150311, 9592)

In [2]:
tfidf_df.sum().sort_values(ascending=False).head(16).index

NameError: name 'tfidf_df' is not defined

In [1]:
from sklearn.model_selection import train_test_split

df = vid_stats_df[['views']].merge(tfidf_df, left_index=True, right_index=True)

X = df.drop('views', axis=1)
y = df['views']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2,
                                                    random_state=42)

X_train.shape, X_test.shape, y_train.shape, y_test.shape

NameError: name 'vid_stats_df' is not defined

In [ ]:
%%time

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

model = LinearRegression()

model.fit(X_train, y_train)

y_pred = model.predict(X_test)

mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"Mean Squared Error: {mse}")
print(f"R-squared: {r2}")

In [ ]:
%%time

from sklearn.linear_model import Lasso
from sklearn.metrics import mean_squared_error, r2_score

lasso_model = Lasso()

lasso_model.fit(X_train, y_train)

y_pred = lasso_model.predict(X_test)

mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"Mean Squared Error: {mse}")
print(f"R-squared: {r2}")

# get lasso coefs
coefs = pd.Series(lasso_model.coef_, index=X.columns)

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score

rf_model = RandomForestRegressor(n_estimators=16, verbose=2, n_jobs=-1)

rf_model.fit(X_train, y_train)

y_pred = rf_model.predict(X_test)

mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"Mean Squared Error: {mse}")
print(f"R-squared: {r2}")

# get feature importance

feature_importances = pd.Series(rf_model.feature_importances_, index=X.columns)

In [ ]:
# join coefs and importances

features = pd.concat([coefs, feature_importances], axis=1)
features.columns = ['lasso', 'rf']
features['score'] = features['lasso'] * features['rf']
features = features[features['score'] != 0]
features.sort_values('score', ascending=False)

In [ ]:
from xgboost import XGBRegressor

model = XGBRegressor(n_estimators=32, verbose=10, n_jobs=-1)

filtered_X_train = X_train[features.index]
filtered_X_test = X_test[features.index]

model.fit(filtered_X_train, y_train)

y_pred = model.predict(filtered_X_test)

mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"Mean Squared Error: {mse}")
print(f"R-squared: {r2}")

In [ ]:
test_data = vid_stats_df.loc[X_test.index]

keep_cols = ['title', 'duration', 'views']

test_data = test_data[keep_cols]

test_data['pred'] = y_pred

test_data.sort_values('pred', ascending=False)